In [ ]:
# ===== Robust Alpha Vantage loader + CHF risk functions =====
# Dependencies: requests, pandas, numpy, pathlib

import os, io, time, json, hashlib, pathlib
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY = os.getenv("api_key")

CACHE_DIR = pathlib.Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

# ---- Schema requirements (minimum columns we rely on) ----
REQ = {
    "TIME_SERIES_DAILY": {"timestamp", "close"},
    "TIME_SERIES_DAILY_ADJUSTED": {"timestamp", "close"},  # adjusted_close is optional preference
    "FX_DAILY": {"timestamp", "close"},
}

# ----------------- helpers (kept in full) -----------------

def _cache_path(function: str, key: str) -> pathlib.Path:
    safe_key = key.replace("/", "_").replace(":", "_").replace(" ", "_")
    fname = f"{function}_{safe_key}.csv"
    return CACHE_DIR / fname

def _is_fresh(path: pathlib.Path, ttl_hours: int) -> bool:
    if not path.exists():
        return False
    age_hours = (time.time() - path.stat().st_mtime) / 3600
    # print(f"Cache file {path} age: {age_hours:.2f} hours (TTL {ttl_hours} hours)")
    return age_hours <= ttl_hours

def _looks_like_json(payload: bytes) -> bool:
    return payload.lstrip()[:1] in (b"{", b"[")

def _valid_columns(df: pd.DataFrame, required: set[str]) -> bool:
    cols = {c.strip().lower() for c in df.columns}
    return required.issubset(cols)

def fetch_csv_robust(url: str, function: str, key: str, ttl_hours: int = 24) -> pd.DataFrame:
    """
    Robust CSV fetch with:
      - on-disk cache (TTL),
      - JSON throttle/error detection (does NOT overwrite cache),
      - schema validation against REQ[function],
      - atomic write on success.
    Returns a DataFrame with lowercase column names.
    """
    path = _cache_path(function, key)

    # Serve fresh cache if valid
    if _is_fresh(path, ttl_hours):
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        if not _valid_columns(df, REQ[function]):
            raise RuntimeError(f"Cached file schema mismatch for {function}:{key}. Columns={list(df.columns)}")
        return df

    # Pull from network
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content

    # Detect JSON throttle/error; do not poison cache
    if _looks_like_json(raw):
        # Try to show a concise message
        try:
            msg = json.loads(raw.decode("utf-8", errors="ignore"))
        except Exception:
            msg = {"body_head": raw[:200].decode("utf-8", errors="ignore")}
        raise RuntimeError(f"Alpha Vantage returned JSON (throttle/error) for {function}:{key} -> {str(msg)[:180]}")

    # Parse CSV and normalize
    df = pd.read_csv(io.BytesIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    if not _valid_columns(df, REQ[function]):
        raise RuntimeError(f"Unexpected CSV schema for {function}:{key}. Got {list(df.columns)}")

    # Atomic-ish write
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        f.write(raw)
    os.replace(tmp, path)
    return df

def _pick_close_column(df: pd.DataFrame) -> pd.Series:
    # Prefer adjusted_close when available (better for total-return style).
    # Fallback to close otherwise.
    if "adjusted_close" in df.columns:
        return df["adjusted_close"]
    return df["close"]

# --------------- public API: data assembly ----------------

def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 252,
    use_adjusted: bool = True,
    ttl_hours: int = 24,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

    holdings: list of dicts with keys:
      - name: row/column label in outputs
      - symbol: Alpha Vantage symbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - value_chf: float; current position value in CHF (for weights)

    Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """
    # Pre-fetch FX once per currency (excluding CHF)
    fx_map: dict[str, pd.Series] = {}
    needed_ccy = sorted({h["ccy"].upper() for h in holdings if h["ccy"].upper() != "CHF"})
    for ccy in needed_ccy:
        url_fx = (
            "https://www.alphavantage.co/query"
            f"?function=FX_DAILY&from_symbol={ccy}&to_symbol=CHF"
            f"&apikey={API_KEY}&outputsize=full&datatype=csv"
        )
        fx_df = fetch_csv_robust(url_fx, "FX_DAILY", f"{ccy}CHF", ttl_hours=ttl_hours)
        fx_df["timestamp"] = pd.to_datetime(fx_df["timestamp"], utc=True)
        fx_s = fx_df.sort_values("timestamp").set_index("timestamp")["close"].rename(f"{ccy}CHF")
        fx_map[ccy] = fx_s
    fx_map_df = pd.DataFrame(fx_map)

    # Build CHF close series per asset
    chf_close = {}
    for h in holdings:
        name  = h["name"]
        sym   = h["symbol"]
        ccy   = h["ccy"].upper()
        gbx   = bool(h.get("gbx", False))

        fn = "TIME_SERIES_DAILY_ADJUSTED" if use_adjusted else "TIME_SERIES_DAILY"
        url_px = (
            "https://www.alphavantage.co/query"
            f"?function={fn}&symbol={sym}"
            f"&apikey={API_KEY}&outputsize=full&datatype=csv"
        )
        px_df = fetch_csv_robust(url_px, fn, sym, ttl_hours=ttl_hours)
        px_df["timestamp"] = pd.to_datetime(px_df["timestamp"], utc=True)
        px_df = px_df.sort_values("timestamp").set_index("timestamp")

        close_local = _pick_close_column(px_df)
        if gbx:
            # LSE pence → GBP. (Does not change returns, only magnitude.)
            close_local = close_local / 100.0

        if ccy == "CHF":
            close_chf = close_local.rename(name)
        else:
            fx = fx_map[ccy]
            df = pd.concat([close_local.rename("p"), fx.rename("x")], axis=1).dropna()
            close_chf = (df["p"] * df["x"]).rename(name)

        chf_close[name] = close_chf

    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(chf_close).dropna()
    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    # Weights (CHF)
    total_val = sum(float(h["value_chf"]) for h in holdings)
    w = pd.Series({h["name"]: float(h["value_chf"]) / total_val for h in holdings})
    w = w.reindex(rets_df.columns).fillna(0.0)

    return rets_df, prices_df, w

# --------------- public API: portfolio risk ----------------

def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [69]:
# ===== Robust Alpha Vantage loader + CHF risk functions =====
# Dependencies: requests, pandas, numpy, pathlib

import os, io, time, json, hashlib, pathlib
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY = os.getenv("api_key")

CACHE_DIR = pathlib.Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

# ---- Schema requirements (minimum columns we rely on) ----
REQ = {
    "TIME_SERIES_DAILY": {"timestamp", "close"},
    "TIME_SERIES_DAILY_ADJUSTED": {"timestamp", "close"},  # adjusted_close is optional preference
    "FX_DAILY": {"timestamp", "close"},
}

# ----------------- helpers (kept in full) -----------------

def _cache_path(function: str, key: str) -> pathlib.Path:
    safe_key = key.replace("/", "_").replace(":", "_").replace(" ", "_")
    fname = f"{function}_{safe_key}.csv"
    return CACHE_DIR / fname

def _is_fresh(path: pathlib.Path, ttl_hours: int) -> bool:
    if not path.exists():
        return False
    age_hours = (time.time() - path.stat().st_mtime) / 3600
    # print(f"Cache file {path} age: {age_hours:.2f} hours (TTL {ttl_hours} hours)")
    return age_hours <= ttl_hours

def _looks_like_json(payload: bytes) -> bool:
    return payload.lstrip()[:1] in (b"{", b"[")

def _valid_columns(df: pd.DataFrame, required: set[str]) -> bool:
    cols = {c.strip().lower() for c in df.columns}
    return required.issubset(cols)

def fetch_csv_robust(url: str, function: str, key: str, ttl_hours: int = 24) -> pd.DataFrame:
    """
    Robust CSV fetch with:
      - on-disk cache (TTL),
      - JSON throttle/error detection (does NOT overwrite cache),
      - schema validation against REQ[function],
      - atomic write on success.
    Returns a DataFrame with lowercase column names.
    """
    path = _cache_path(function, key)

    # Serve fresh cache if valid
    if _is_fresh(path, ttl_hours):
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        if not _valid_columns(df, REQ[function]):
            raise RuntimeError(f"Cached file schema mismatch for {function}:{key}. Columns={list(df.columns)}")
        return df

    # Pull from network
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content

    # Detect JSON throttle/error; do not poison cache
    if _looks_like_json(raw):
        # Try to show a concise message
        try:
            msg = json.loads(raw.decode("utf-8", errors="ignore"))
        except Exception:
            msg = {"body_head": raw[:200].decode("utf-8", errors="ignore")}
        raise RuntimeError(f"Alpha Vantage returned JSON (throttle/error) for {function}:{key} -> {str(msg)[:180]}")

    # Parse CSV and normalize
    df = pd.read_csv(io.BytesIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    if not _valid_columns(df, REQ[function]):
        raise RuntimeError(f"Unexpected CSV schema for {function}:{key}. Got {list(df.columns)}")

    # Atomic-ish write
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        f.write(raw)
    os.replace(tmp, path)
    return df

def _pick_close_column(df: pd.DataFrame) -> pd.Series:
    # Prefer adjusted_close when available (better for total-return style).
    # Fallback to close otherwise.
    if "adjusted_close" in df.columns:
        return df["adjusted_close"]
    return df["close"]

# --------------- public API: data assembly ----------------

def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 252,
    use_adjusted: bool = True,
    ttl_hours: int = 24,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

    holdings: list of dicts with keys:
      - name: row/column label in outputs
      - symbol: Alpha Vantage symbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - value_chf: float; current position value in CHF (for weights)

    Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """
    # Pre-fetch FX once per currency (excluding CHF)
    fx_map: dict[str, pd.Series] = {}
    needed_ccy = sorted({h["ccy"].upper() for h in holdings if h["ccy"].upper() != "CHF"})
    for ccy in needed_ccy:
        url_fx = (
            "https://www.alphavantage.co/query"
            f"?function=FX_DAILY&from_symbol={ccy}&to_symbol=CHF"
            f"&apikey={API_KEY}&outputsize=full&datatype=csv"
        )
        fx_df = fetch_csv_robust(url_fx, "FX_DAILY", f"{ccy}CHF", ttl_hours=ttl_hours)
        fx_df["timestamp"] = pd.to_datetime(fx_df["timestamp"], utc=True)
        fx_s = fx_df.sort_values("timestamp").set_index("timestamp")["close"].rename(f"{ccy}CHF")
        fx_map[ccy] = fx_s
    fx_map_df = pd.DataFrame(fx_map)

    # Build CHF close series per asset
    chf_close = {}
    for h in holdings:
        name  = h["name"]
        sym   = h["symbol"]
        ccy   = h["ccy"].upper()
        gbx   = bool(h.get("gbx", False))

        fn = "TIME_SERIES_DAILY_ADJUSTED" if use_adjusted else "TIME_SERIES_DAILY"
        url_px = (
            "https://www.alphavantage.co/query"
            f"?function={fn}&symbol={sym}"
            f"&apikey={API_KEY}&outputsize=full&datatype=csv"
        )
        px_df = fetch_csv_robust(url_px, fn, sym, ttl_hours=ttl_hours)
        px_df["timestamp"] = pd.to_datetime(px_df["timestamp"], utc=True)
        px_df = px_df.sort_values("timestamp").set_index("timestamp")

        close_local = _pick_close_column(px_df)
        if gbx:
            # LSE pence → GBP. (Does not change returns, only magnitude.)
            close_local = close_local / 100.0

        if ccy == "CHF":
            close_chf = close_local.rename(name)
        else:
            fx = fx_map[ccy]
            df = pd.concat([close_local.rename("p"), fx.rename("x")], axis=1).dropna()
            close_chf = (df["p"] * df["x"]).rename(name)

        chf_close[name] = close_chf
        # print(f'chf_close["{name}"] tail:\n{chf_close[name].tail()}')
        # print the last value in 'chf_close[name]
    # print(f'Last value: {chf_close}')



    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(chf_close).dropna()
    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(2)).dropna()

    values = {}
    for h in holdings:
        name = h['name']
        last_price = chf_close[name].iloc[-1]
        print(h['name'], h['position'], last_price)

        values[name] = h['position'] * last_price
    print(values)

    # Weights (CHF)
    total_val = sum(float(h["value_chf"]) for h in holdings)
    w = pd.Series({h["name"]: float(h["value_chf"]) / total_val for h in holdings})
    w = w.reindex(rets_df.columns).fillna(0.0)

    return rets_df, prices_df, w

# --------------- public API: portfolio risk ----------------

def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [70]:
holdings0 = [
    {"name":"Unilever", "symbol":"ULVR.LON", "ccy":"GBP", "gbx":True,  "value_chf": 25000},
    {"name":"Shell",    "symbol":"SHEL.LON", "ccy":"GBP", "gbx":True,  "value_chf": 13000},
    {"name":"NatWest",  "symbol":"NWG.LON",  "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Barclays", "symbol":"BARC.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Tesco",    "symbol":"TSCO.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"SWDA",     "symbol":"SWDA.LON", "ccy":"GBP", "gbx":True,  "value_chf":  12000},
    {"name":"EMIM",     "symbol":"EMIM.LON", "ccy":"GBP", "gbx":True,  "value_chf":  8000},
    {"name":"IBM",      "symbol":"IBM",      "ccy":"USD", "gbx":False, "value_chf":  4000},
    {"name":"ERNS",     "symbol":"ERNS.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
]
holdings = [
    {"name":"Unilever", "symbol":"ULVR.LON", "ccy":"GBP", "gbx":True, "position": 1266, "value_chf": 25000},
    {"name":"Shell",    "symbol":"SHEL.LON", "ccy":"GBP", "gbx":True, "position": 3, "value_chf": 13000},
    {"name":"NatWest",  "symbol":"NWG.LON",  "ccy":"GBP", "gbx":True, "position": 3, "value_chf":  5000},
    {"name":"Barclays", "symbol":"BARC.LON", "ccy":"GBP", "gbx":True, "position": 3, "value_chf":  5000},
    {"name":"Tesco",    "symbol":"TSCO.LON", "ccy":"GBP", "gbx":True, "position": 3, "value_chf":  5000},
    {"name":"SWDA",     "symbol":"SWDA.LON", "ccy":"GBP", "gbx":True, "position": 3, "value_chf":  12000},
    {"name":"EMIM",     "symbol":"EMIM.LON", "ccy":"GBP", "gbx":True, "position": 3, "value_chf":  8000},
    {"name":"IBM",      "symbol":"IBM",      "ccy":"USD", "gbx":False, "position": 3, "value_chf":  4000},
    {"name":"ERNS",     "symbol":"ERNS.LON", "ccy":"GBP", "gbx":True, "position": 3, "value_chf":  5000},
]

TTL = 100
PERIOD = 252

rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=PERIOD, use_adjusted=False, ttl_hours=TTL)

# print(f'rets_df: {rets_df.tail()}')
# print(f'prices_df: {prices_df.tail()}')

risk = portfolio_risk(rets_df, w)

# print(rets_df.tail())
print("Portfolio σ (annualized, CHF): {:.2%}".format(risk["port_vol"]))
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
# Optional:
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')
print(f'COVARIANCE:')
print(f'{risk["cov_annual"]}')

Unilever 1266 50.5137024
Shell 3 29.168358400000002
NatWest 3 6.093716479999999
Barclays 3 4.07891104
Tesco 3 4.61904576
SWDA 3 98.60032000000001
EMIM 3 33.101536
IBM 3 193.986717
ERNS 3 1.099122688
{'Unilever': np.float64(63950.3472384), 'Shell': np.float64(87.50507520000001), 'NatWest': np.float64(18.28114944), 'Barclays': np.float64(12.23673312), 'Tesco': np.float64(13.85713728), 'SWDA': np.float64(295.80096000000003), 'EMIM': np.float64(99.304608), 'IBM': np.float64(581.960151), 'ERNS': np.float64(3.297368064)}
Portfolio σ (annualized, CHF): 20.28%
          Weight  Vol_1Y_CHF    MRC  PRC_%
Unilever   0.305       0.255  0.157   23.5
Shell      0.159       0.367  0.274   21.4
SWDA       0.146       0.251  0.197   14.2
Barclays   0.061       0.483  0.364   11.0
EMIM       0.098       0.257  0.197    9.5
NatWest    0.061       0.414  0.288    8.7
IBM        0.049       0.429  0.211    5.1
Tesco      0.061       0.346  0.149    4.5
ERNS       0.061       0.115  0.073    2.2
CORRELATION